# AGRIVISION - Plant Detector Training

Trains a custom single-class (`plant`) object detector to replace the generic
COCO-SSD model in the `cv-prototype` demo.

## Dataset (Open Images V7)

**POSITIVES - whole-organism plant classes only:** `Plant`, `Houseplant`,
`Tree`, `Palm tree`, all merged into one `plant` class. One box = one whole
plant.
`Flower`, `Flowerpot`, `Fruit`, `Vegetable` are deliberately **excluded**.
Open Images boxes those as separate sub-objects (a single bloom, an empty pot,
one tomato, a pile of produce). Merging them into `plant` is what taught the
previous model that any round / potted / produce-like blob is a plant - which
is why it reported a human face/body as `plant` at ~93% confidence.

**NEGATIVES - background images with no plant at all:** people, faces, hands,
vehicles, furniture, buildings, indoor scenes, empty pots. Exported with empty
label files so YOLOv8 learns them as explicit "nothing to detect here"
examples. The rover/greenhouse camera sees mostly non-plants, so the detector
has to be trained to stay silent on them.

Fine-tunes **YOLOv8n**, then exports to **TensorFlow Lite** (ONNX + onnx2tf)
for the browser demo's `tfjs-tflite` runtime.

**Before running:** Colab `Runtime > Change runtime type` -> **GPU** (T4 is
fine), then `Runtime > Run all`. At the end you download a zip - unzip
`plant_detector.tflite` + `metadata.json` into `cv-prototype/model/`.

In [ ]:
!pip install -q fiftyone ultralytics

## 1. Download the POSITIVE set - whole plants only

Whole-organism classes only. Raise `MAX_SAMPLES_PER_CLASS` for more data
(more samples = better accuracy, longer download + training).

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# Whole-organism plant classes ONLY. Do NOT add Flower / Flowerpot / Fruit /
# Vegetable: Open Images boxes those as separate sub-objects, and merging them
# into `plant` is what made the old model fire on faces, bodies and pots.
PLANT_CLASSES = ["Plant", "Houseplant", "Tree", "Palm tree"]

# Broad "a plant is present" vocabulary - used ONLY to scrub the negative set
# below so no real plant leaks in. Keep it generous.
PLANT_VOCAB = {
    "Plant", "Houseplant", "Tree", "Palm tree", "Flower", "Flowerpot",
    "Fruit", "Vegetable", "Plant stem", "Leaf", "Bush", "Shrub", "Herb",
    "Grass", "Willow tree", "Maple", "Pine", "Christmas tree", "Rose",
    "Sunflower", "Common sunflower", "Lavender", "Lily", "Tomato",
}

MAX_SAMPLES_PER_CLASS = 2000   # ~8k positive images across the 4 classes

pos_train = foz.load_zoo_dataset(
    "open-images-v7", split="train", label_types=["detections"],
    classes=PLANT_CLASSES, max_samples=MAX_SAMPLES_PER_CLASS,
    dataset_name="agrivision-pos-train", shuffle=True,
)
pos_val = foz.load_zoo_dataset(
    "open-images-v7", split="validation", label_types=["detections"],
    classes=PLANT_CLASSES, max_samples=max(300, MAX_SAMPLES_PER_CLASS // 5),
    dataset_name="agrivision-pos-val", shuffle=True,
)
print(pos_train)
print(pos_val)

## 1b. Download the NEGATIVE / background set - images with NO plants

A detector that has only ever seen images containing plants will fire on
anything blob-shaped. These images are exported with **empty** label files so
YOLOv8 learns them as hard negatives. Any downloaded image that actually
contains a plant is *discarded* (not blanked), so we never label a real plant
as background.

In [ ]:
# What a field / greenhouse / rover camera sees that is NOT a plant.
NEGATIVE_CLASSES = [
    "Person", "Human face", "Human head", "Human body", "Human hand",
    "Human arm", "Man", "Woman", "Boy", "Girl", "Clothing",
    "Car", "Truck", "Wheel", "Tire", "Bicycle",
    "Building", "House", "Door", "Window", "Brick",
    "Chair", "Table", "Desk", "Couch", "Bed", "Cabinetry", "Shelf",
    "Bottle", "Box", "Bucket", "Flowerpot",   # an EMPTY pot is not a plant
    "Dog", "Cat", "Bird",
]

MAX_NEGATIVES_TRAIN = 4000   # ~half the positive count -> strong negative signal
MAX_NEGATIVES_VAL   = 800

def load_negatives(split, max_samples, name):
    ds = foz.load_zoo_dataset(
        "open-images-v7", split=split, label_types=["detections"],
        classes=NEGATIVE_CLASSES,
        max_samples=max_samples * 2,   # over-pull; we drop any plant images
        dataset_name=name, shuffle=True,
    )
    keep = []
    for s in ds.iter_samples(progress=True):
        dets = s.ground_truth.detections if s.ground_truth else []
        if any(d.label in PLANT_VOCAB for d in dets):
            continue                       # a plant snuck in - discard
        s["ground_truth"] = fo.Detections(detections=[])   # background image
        s.save()
        keep.append(s.id)
        if len(keep) >= max_samples:
            break
    clean = ds.select(keep).clone(name + "-clean")
    print(f"{name}: kept {len(clean)} plant-free background images")
    return clean

neg_train = load_negatives("train", MAX_NEGATIVES_TRAIN, "agrivision-neg-train")
neg_val   = load_negatives("validation", MAX_NEGATIVES_VAL, "agrivision-neg-val")

## 2. Merge plant labels into one `plant` class, add negatives, export as YOLO

Every positive box (Plant / Houseplant / Tree / Palm tree) becomes class
`plant`; negative images carry zero boxes. Both go into one YOLOv5-format
dataset - Ultralytics reads images with empty/missing label files as
background.

In [ ]:
def remap_to_plant(dataset):
    for sample in dataset.iter_samples(progress=True, autosave=True):
        dets = sample.ground_truth
        if dets is None:
            continue
        for det in dets.detections:
            det.label = "plant"

remap_to_plant(pos_train)
remap_to_plant(pos_val)

EXPORT_DIR = "/content/plant_yolo"

def export_split(pos, neg, split):
    combined = pos.clone(f"agrivision-{split}-combined")
    combined.add_samples([s.copy() for s in neg])
    combined.export(
        export_dir=EXPORT_DIR,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
        split=split,
        classes=["plant"],
    )
    n_bg = sum(1 for s in combined
               if s.ground_truth is None or len(s.ground_truth.detections) == 0)
    print(f"{split}: {len(combined)} images "
          f"({n_bg} background / {len(combined) - n_bg} with plants)")

export_split(pos_train, neg_train, "train")
export_split(pos_val,   neg_val,   "val")
print("Exported to", EXPORT_DIR)

## 3. Train YOLOv8n on the plant dataset

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

# The dataset now contains background images (empty label files) as hard
# negatives, so PRECISION matters as much as recall. Watch metrics/precision(B)
# in the training log - if it stays low, pull more negatives in cell 1b.
results = model.train(
    data=f"{EXPORT_DIR}/dataset.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,
    project="agrivision",
    name="plant_detector",
)

In [ ]:
metrics = model.val()
# mAP50, mAP50-95, plus precision/recall on the held-out set (incl. negatives).
print("mAP50 =", metrics.box.map50, " mAP50-95 =", metrics.box.map)
print("precision =", metrics.box.mp, " recall =", metrics.box.mr)

## 4. Export to TensorFlow Lite for the browser demo

In [ ]:
!pip install -q onnx onnx2tf onnx_graphsurgeon sng4onnx onnxsim onnxruntime

best_weights = "agrivision/plant_detector/weights/best.pt"
export_model = YOLO(best_weights)

# Export to ONNX first, then onnx2tf converts it to TFLite below. NOTE: verified
# by parsing the shipped plant_detector.tflite directly (via the `tflite`
# flatbuffer schema) that onnx2tf's default output for this model keeps the ONNX
# (NCHW) input layout -> [1, 3, imgsz, imgsz], NOT channels-last (NHWC), and that
# the box coords in the [1, 5, 8400] output are NORMALIZED to [0,1] (the graph
# ends with xywh_pixels * 1/imgsz). script.js's CustomModel.detect() feeds NCHW
# and auto-detects the normalized coords. If you change onnx2tf's flags/version,
# re-check both before assuming the browser side still lines up.
onnx_path = export_model.export(format="onnx", imgsz=640, opset=12)

In [ ]:
import json, shutil, glob, subprocess

TFLITE_OUT_DIR = "plant_detector_tflite"

subprocess.run([
    "onnx2tf", "-i", str(onnx_path), "-o", TFLITE_OUT_DIR,
    "-osd",  # output only the plain float32 saved_model/tflite, skip quantized variants
], check=True)

tflite_path = glob.glob(f"{TFLITE_OUT_DIR}/*_float32.tflite")[0]
shutil.copy(tflite_path, f"{TFLITE_OUT_DIR}/plant_detector.tflite")

# Small metadata.json the browser app reads directly (avoids parsing the yaml
# ultralytics also emits) so script.js knows the class list and input size.
with open(f"{TFLITE_OUT_DIR}/metadata.json", "w") as f:
    json.dump({"names": ["plant"], "imgsz": 640}, f)

shutil.make_archive("plant_detector_tflite", "zip", TFLITE_OUT_DIR, base_dir=".")
print("Ready: plant_detector_tflite.zip (plant_detector.tflite + metadata.json)")

In [ ]:
from google.colab import files
files.download("plant_detector_tflite.zip")

## 5. Install into the demo

Unzip `plant_detector_tflite.zip` and copy `plant_detector.tflite` and
`metadata.json` into `cv-prototype/model/`, then reload the AGRIVISION page.
`script.js` checks for `model/plant_detector.tflite` on startup, auto-detects
normalized-vs-pixel box coords, and falls back to COCO-SSD if the file is
missing.

**Sanity-check before trusting it:** point the camera at a person with no plant
in frame and watch the console - `[AGRIVISION] [COUNT] ... FINAL PLANTS=0` is
the pass condition. Then check one / two / three separated plants give 1 / 2 / 3.